# 04 - Modeling
**Lab AUEC Clustering | Defensa Papers ML UCB 2026**

---
## Objetivos (CRISP-DM: Modeling)
- Entrenar tres baselines (B1, B2, B3) como punto de referencia
- Entrenar el Autoencoder Convolucional (CAE) con loss MSE
- Extraer representaciones del espacio latente
- Aplicar UMAP sobre el espacio latente
- Ejecutar K-means y DBSCAN sobre las representaciones AUEC
- Persistir etiquetas y modelos para evaluacion en notebook 05

In [ ]:
import sys, os
sys.path.insert(0, '..')
os.chdir('..')

import numpy as np
import matplotlib.pyplot as plt

from src.data_loader   import load_config
from src.preprocessing import load_processed
from src.autoencoder   import build_autoencoder, train_autoencoder, save_encoder_weights
from src.clustering    import (run_kmeans, run_dbscan, run_pca, run_umap,
                               evaluar_clustering)
from src.utils import (set_plot_style, save_fig, print_hallazgo,
                       print_separador, resultados_a_csv, COLORS)

set_plot_style()
cfg  = load_config()
SEED = cfg["seed"]
K    = cfg["kmeans"]["k"]

In [ ]:
arrays = load_processed(
    ["x_train_cnn", "x_test_cnn", "x_train_flat", "x_test_flat", "y_train", "y_test"],
    cfg["data"]["processed_dir"]
)
x_train_cnn  = arrays["x_train_cnn"]
x_test_cnn   = arrays["x_test_cnn"]
x_train_flat = arrays["x_train_flat"]
x_test_flat  = arrays["x_test_flat"]
y_train      = arrays["y_train"]
y_test       = arrays["y_test"]

print(f"Datos cargados - train_flat: {x_train_flat.shape}  train_cnn: {x_train_cnn.shape}")

## 4.1 Baseline B1 - K-means sobre pixeles (784D)

In [ ]:
print_separador("B1: K-means raw (784D)")
km_b1, ltr_b1, lte_b1 = run_kmeans(
    x_train_flat, x_test_flat, k=K,
    n_init=cfg["kmeans"]["n_init"], seed=SEED
)
res_b1 = evaluar_clustering(x_train_flat, ltr_b1, y_train, "B1: KMeans raw (784D)")
print(res_b1)

print_hallazgo("B1", [
    f"ACC = {res_b1.get('ACC ↑', 'N/A')}",
    "K-means en alta dimension sufre curse of dimensionality",
    "Distancias euclidianas poco informativas en 784D"
])

## 4.2 Baseline B2 - PCA(50) + K-means

In [ ]:
print_separador("B2: PCA(50) + K-means")
pca, x_train_pca, x_test_pca = run_pca(
    x_train_flat, x_test_flat,
    n_components=cfg["pca"]["n_components"], seed=SEED
)
km_b2, ltr_b2, lte_b2 = run_kmeans(
    x_train_pca, x_test_pca, k=K,
    n_init=cfg["kmeans"]["n_init"], seed=SEED
)
res_b2 = evaluar_clustering(x_train_pca, ltr_b2, y_train, "B2: PCA(50)+KMeans")
print(res_b2)

var_ret = pca.explained_variance_ratio_.sum() * 100
print_hallazgo("B2", [
    f"PCA(50) retiene {var_ret:.1f}% de varianza",
    f"ACC = {res_b2.get('ACC ↑', 'N/A')}",
    "Mejora leve sobre B1: reduccion lineal ayuda poco"
])

## 4.3 Baseline B3 - UMAP directo sobre pixeles + K-means

In [ ]:
print_separador("B3: UMAP(raw 784D) + K-means")
umap_b3, x_train_ub3, x_test_ub3 = run_umap(
    x_train_flat, x_test_flat,
    n_components=cfg["umap"]["n_components"],
    n_neighbors=cfg["umap"]["n_neighbors"],
    min_dist=cfg["umap"]["min_dist"], seed=SEED
)
km_b3, ltr_b3, lte_b3 = run_kmeans(
    x_train_ub3, x_test_ub3, k=K,
    n_init=cfg["kmeans"]["n_init"], seed=SEED
)
res_b3 = evaluar_clustering(x_train_ub3, ltr_b3, y_train, "B3: UMAP(raw)+KMeans")
print(res_b3)

print_hallazgo("B3", [
    f"ACC = {res_b3.get('ACC ↑', 'N/A')}",
    "UMAP no lineal mejora significativamente sobre PCA",
    "Captura estructura de variedad (manifold) de los digitos"
])

## 4.4 Autoencoder Convolucional (CAE)

In [ ]:
print_separador("CAE - Arquitectura")
autoencoder, encoder = build_autoencoder(
    latent_dim=cfg["ae"]["latent_dim"],
    input_shape=(28, 28, 1)
)
autoencoder.summary()

In [ ]:
print_separador("CAE - Entrenamiento")
history = train_autoencoder(autoencoder, x_train_cnn, cfg, validation_split=0.1)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(history.history["loss"],     label="Train loss",      color=COLORS["primary"])
ax.plot(history.history["val_loss"], label="Validacion loss", color=COLORS["accent"])
ax.set_xlabel("Epoca")
ax.set_ylabel("MSE")
ax.set_title("Curva de aprendizaje - Autoencoder Convolucional")
ax.legend()
plt.tight_layout()
save_fig("04_curva_aprendizaje_ae")
plt.show()

final_tr  = history.history["loss"][-1]
final_val = history.history["val_loss"][-1]
print_hallazgo("CAE", [
    f"MSE train final   = {final_tr:.6f}",
    f"MSE val   final   = {final_val:.6f}",
    f"Epocas entrenadas = {len(history.history['loss'])}"
])

## 4.5 Calidad de reconstruccion

In [ ]:
recon_test = autoencoder.predict(x_test_cnn[:10], verbose=0)

fig, axes = plt.subplots(2, 10, figsize=(14, 3))
for i in range(10):
    axes[0, i].imshow(x_test_cnn[i, :, :, 0], cmap="gray")
    axes[0, i].axis("off")
    axes[1, i].imshow(recon_test[i, :, :, 0],  cmap="gray")
    axes[1, i].axis("off")
axes[0, 0].set_ylabel("Original",     fontsize=9, rotation=0, labelpad=40)
axes[1, 0].set_ylabel("Reconstruida", fontsize=9, rotation=0, labelpad=40)
fig.suptitle("Reconstruccion del Autoencoder (test set)", fontsize=12)
plt.tight_layout()
save_fig("04_reconstruccion_ae")
plt.show()

In [ ]:
save_encoder_weights(encoder, cfg["output"]["models"])

## 4.6 Espacio latente del AE -> UMAP -> K-means (AUEC)

In [ ]:
print_separador("AUEC - Extraccion del espacio latente")
z_train = encoder.predict(x_train_cnn, verbose=0)
z_test  = encoder.predict(x_test_cnn,  verbose=0)
print(f"Espacio latente - train: {z_train.shape}  test: {z_test.shape}")

print_hallazgo("LATENTE", [
    f"Dimensiones: {z_train.shape[1]}D  (vs 784D original)",
    f"Rango train: [{z_train.min():.3f}, {z_train.max():.3f}]",
    "El AE comprime 784D -> 10D conservando estructura visual"
])

In [ ]:
print_separador("AUEC - UMAP sobre espacio latente")
umap_ae, z_train_u, z_test_u = run_umap(
    z_train, z_test,
    n_components=cfg["umap"]["n_components"],
    n_neighbors=cfg["umap"]["n_neighbors"],
    min_dist=cfg["umap"]["min_dist"], seed=SEED
)
print(f"UMAP(AE latente) - train: {z_train_u.shape}  test: {z_test_u.shape}")

In [ ]:
print_separador("AUEC - K-means sobre UMAP(latente)")
km_auec, ltr_auec, lte_auec = run_kmeans(
    z_train_u, z_test_u, k=K,
    n_init=cfg["kmeans"]["n_init"], seed=SEED
)
res_auec_km = evaluar_clustering(z_train_u, ltr_auec, y_train, "AUEC: AE+UMAP+KMeans")
print(res_auec_km)

print_hallazgo("AUEC KMeans", [
    f"ACC = {res_auec_km.get('ACC ↑', 'N/A')}",
    "Combinacion AE+UMAP supera a cada tecnica por separado",
    "El AE aprende representacion invariante a variaciones"
])

## 4.7 DBSCAN sobre espacio AUEC (2D)

In [ ]:
print_separador("AUEC - DBSCAN sobre UMAP(latente) 2D")
_, z_train_2d, z_test_2d = run_umap(
    z_train, z_test,
    n_components=2,
    n_neighbors=cfg["umap"]["viz"]["n_neighbors"],
    min_dist=cfg["umap"]["viz"]["min_dist"], seed=SEED
)

db_auec, ltr_dbscan = run_dbscan(
    z_train_2d,
    eps=cfg["dbscan"]["eps"],
    min_samples=cfg["dbscan"]["min_samples"]
)
res_auec_db = evaluar_clustering(z_train_2d, ltr_dbscan, y_train, "AUEC: AE+UMAP+DBSCAN")
print(res_auec_db)

n_noise = (ltr_dbscan == -1).sum()
print_hallazgo("DBSCAN", [
    f"Clusters encontrados: {res_auec_db['N clusters']}  (esperados: 10)",
    f"Ruido: {n_noise} puntos ({n_noise/len(ltr_dbscan)*100:.1f}%)",
    "DBSCAN es sensible a eps; eps=0.5 tiende a unir algunos clusters"
])

## 4.8 Persistencia de etiquetas y representaciones

In [ ]:
import pathlib
pathlib.Path("data/processed").mkdir(exist_ok=True)

np.save("data/processed/ltr_b1.npy",     ltr_b1)
np.save("data/processed/ltr_b2.npy",     ltr_b2)
np.save("data/processed/ltr_b3.npy",     ltr_b3)
np.save("data/processed/ltr_auec.npy",   ltr_auec)
np.save("data/processed/ltr_dbscan.npy", ltr_dbscan)
np.save("data/processed/z_train.npy",    z_train)
np.save("data/processed/z_train_u.npy",  z_train_u)
np.save("data/processed/z_train_2d.npy", z_train_2d)

print("Etiquetas y representaciones guardadas.")
for name, arr in [("ltr_b1", ltr_b1), ("ltr_auec", ltr_auec),
                   ("z_train", z_train), ("z_train_u", z_train_u), ("z_train_2d", z_train_2d)]:
    print(f"  {name}: {arr.shape}")

## 4.9 Resumen de modelos entrenados

| Modelo | Tecnica | Representacion | Persistido |
|--------|---------|----------------|------------|
| B1 | K-means | 784D raw | labels .npy |
| B2 | PCA+KMeans | 50D PCA | labels .npy |
| B3 | UMAP+KMeans | 10D UMAP | labels .npy |
| CAE | Conv Autoencoder | — | encoder_weights.h5 |
| AUEC-KM | AE+UMAP+KMeans | 10D UMAP(Z) | labels .npy |
| AUEC-DB | AE+UMAP+DBSCAN | 2D UMAP(Z) | labels .npy |

> **Siguiente:** `05_evaluation.ipynb` - Comparacion, visualizaciones y conclusiones.